# revT2V: quick demo

A minimal two-video demo: pick one method, generate it alongside the frozen
teacher from the same initial noise, and compare them with a couple of
`scripts/evaluate.py`'s own metrics -- no sweep over prompts/seeds, no report
assets, just enough to show someone the method working. For the full
multi-method comparison, see `compare_methods.ipynb` instead.

Only two videos are ever generated: **teacher** (forward-time) and
**`METHOD`** (reverse-time, below). `teacher_reversed` used for the
direction-agreement metric is just the teacher's own frames played backwards
-- not a third generation.


In [ ]:
# 1. Setup -- clone/pull, install, HF token, GPU check.
import os
import subprocess
import sys

REPO_URL = "https://github.com/silentlooop/revT2V.git"
REPO_DIR = "/content/revT2V"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)

subprocess.run(["pip", "install", "-q", "-e", ".[eval]"], check=True)

# Colab/Kaggle preinstall a torchao whose quantization API diffusers' lazy
# imports now expect a newer version of -- never used here, so removing it
# avoids the crash. Same fix as the other notebooks.
subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"])

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Add a Colab secret named HF_TOKEN with read access to "
        "silentlooop/revt2v-ckpt and turn on notebook access for it."
    ) from exc

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then Run all again."
print("CUDA:", torch.cuda.get_device_name(0))


In [ ]:
# 2. Pick a method and a prompt.
HF_REPO = "silentlooop/revt2v-ckpt"
PROMPT = "a red ball bouncing on a wooden floor"
SEED = 0

# One of: "conv_lora" (main method), "attn_injection", "attn_rotation"
# (each needs its checkpoint on HF_REPO, loaded below -- falls back to
# zero-shot for attn_injection, errors for the others if missing), or
# "conv_mirror" (training-free, always available).
METHOD = "conv_lora"

GEN = dict(
    negative_prompt="watermark, text",
    num_inference_steps=25,
    guidance_scale=9.0,
    num_frames=16,
    height=256,
    width=256,
)
FPS = 8
OUT_DIR = "results/demo"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# 3. Load the teacher, then METHOD's checkpoint if it needs one.
from revt2v.infer import MethodBank

bank = MethodBank(hf_repo_id=HF_REPO)
bank.pipeline.set_progress_bar_config(disable=True)

if METHOD in ("conv_lora", "attn_injection"):
    loaded = bank.load_adapter(METHOD)
    if not loaded and METHOD != "attn_injection":
        raise RuntimeError(f"no checkpoint found for '{METHOD}' on {HF_REPO} -- train it first or pick another method")
    print(METHOD, "loaded" if loaded else "no checkpoint found, running zero-shot")
elif METHOD == "attn_rotation":
    if not bank.load_attn_rotation_adapter():
        raise RuntimeError(f"no attn_rotation checkpoint found on {HF_REPO} -- train it first or pick another method")
    print(METHOD, "loaded")
elif METHOD == "conv_mirror":
    print(METHOD, "training-free, always available")
else:
    raise ValueError(f"unknown method {METHOD!r}")

assert METHOD in bank.methods, f"'{METHOD}' still unavailable after loading -- see the message above"


## Generate: teacher vs. METHOD, same initial noise

In [ ]:
# 4. The only two generations in this notebook.
from IPython.display import Video, display

from revt2v.utils import _to_uint8_frames, save_video

unet = bank.pipeline.unet


def initial_noise(seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    return torch.randn((1, 4, GEN["num_frames"], GEN["height"] // 8, GEN["width"] // 8),
                        generator=g, device="cuda", dtype=unet.dtype)


z = initial_noise(SEED)
teacher_result = bank.generate(PROMPT, "teacher", latents=z, **GEN)
student_result = bank.generate(PROMPT, METHOD, latents=z, **GEN)

teacher_video = _to_uint8_frames(teacher_result["video"])  # (F, H, W, 3) uint8
student_video = _to_uint8_frames(student_result["video"])

teacher_path = save_video(teacher_video, f"{OUT_DIR}/teacher.mp4", fps=FPS)
student_path = save_video(student_video, f"{OUT_DIR}/{METHOD}.mp4", fps=FPS)

print(f"prompt: {PROMPT}")
print(f"teacher         -> {teacher_path}")
print(f"{METHOD:<15} -> {student_path}")
display(Video(str(teacher_path), embed=True, width=256))
display(Video(str(student_path), embed=True, width=256))


## Metrics: `scripts/evaluate.py`'s CLIP score and optical flow, reused as-is

In [ ]:
# 5. CLIP score (prompt alignment) and optical flow (motion) for both videos,
# plus direction agreement between METHOD and the teacher's own frames played
# backwards (a free derived reference -- not a new generation; a correct
# reverse-time method should move the same way as that, not the teacher
# itself).
import numpy as np

sys.path.insert(0, "scripts")  # scripts/ isn't a package; `evaluate` would also clash with HF's own package
import evaluate as ev

clip_model, clip_processor = ev.load_clip(bank.device)
raft_model = ev.load_raft(bank.device)

clip_teacher = ev.compute_clip_score(teacher_video, PROMPT, clip_model, clip_processor)
clip_student = ev.compute_clip_score(student_video, PROMPT, clip_model, clip_processor)

flow_teacher = ev.optical_flow(teacher_video, raft_model)
flow_student = ev.optical_flow(student_video, raft_model)
mag_teacher = float(np.linalg.norm(flow_teacher, axis=-1).mean())
mag_student = float(np.linalg.norm(flow_student, axis=-1).mean())
motion_ratio = mag_student / max(mag_teacher, 1e-6)
frozen = motion_ratio < 0.3

teacher_reversed = teacher_video[::-1]  # same frames as teacher, just reordered
direction_cosine = ev.compute_flow_metrics(student_video, teacher_reversed, raft_model)["flow_cosine"]

print(f"{'metric':<30}{'teacher':>12}{METHOD:>16}")
print(f"{'CLIP score (x prompt)':<30}{clip_teacher:>12.2f}{clip_student:>16.2f}")
print(f"{'optical flow magnitude':<30}{mag_teacher:>12.3f}{mag_student:>16.3f}")
print()
print(f"motion_ratio ({METHOD} / teacher):        {motion_ratio:.3f}" + ("  <-- looks frozen (<0.3x teacher's motion)" if frozen else ""))
print(f"direction_cosine ({METHOD} vs teacher reversed): {direction_cosine:.3f}  (+1 = matches reverse motion, -1 = opposite, 0 = unrelated)")
